# American Option Pricing with a Binomial Tree

This notebook demonstrates how to price an American option with a discrete-time binomial tree. The implementation works backward from terminal option payoffs and checks early exercise value at each node, which is the key feature that separates American options from European options.

At each time step, the stock price can move up by a factor $u$ or down by a factor $d$:

$$
S_{i,j} = S_0 u^j d^{i-j}
$$

In [1]:
import numpy as np

## Model Inputs

The parameters below define the initial stock price, strike price, time to maturity, risk-free rate, number of time steps, and up/down movement factors. `opttype` can be set to `C` for a call option or `P` for a put option.

The time step is:

$$
\Delta t = \frac{T}{N}
$$

The risk-neutral probability is:

$$
p = \frac{e^{r\Delta t} - d}{u - d}
$$

In [2]:
#initialize parameters

S0 = 100            #initial stock price
K = 100             #strike price
T = 1               #time in years
r = 0.06            #annual risk-free rate
N = 3               #number of time steps
u = 1.1             #up-factor
d = 1/u             #down-factor
opttype = 'C'       #option type, C:call, P:put

## Pricing Function

The function first computes payoffs at maturity, then moves backward through the tree. At every earlier node it compares the discounted continuation value with the immediate exercise value and keeps the larger value.

For a call option, the terminal payoff is:

$$
C_T = \max(S_T - K, 0)
$$

For a put option, the terminal payoff is:

$$
P_T = \max(K - S_T, 0)
$$

The American option value is calculated as:

$$
V_{i,j} = \max\left(\text{exercise value},\ e^{-r\Delta t}\left(pV_{i+1,j+1} + (1-p)V_{i+1,j}\right)\right)
$$

In [4]:
def american_option_tree(S0, K, T, r, N, u, d, opttype = 'P'):
    dt = T/N
    p = (np.exp(r*dt)-d)/(u-d)
    disc = np.exp(-r*dt)
    S = S0*d**np.arange(N, -1, -1)*u**np.arange(0, N+1, 1)
    if opttype == 'P':
        C = np.maximum(K-S, 0)
    else:
        C = np.maximum(S-K, 0)
    for i in range(N-1, -1, -1):
        S = S0 * d**np.arange(i, -1, -1) * u**np.arange(0, i+1, 1)
        C[:i+1] = disc * (p*C[1:i+2] + (1-p)*C[0:i+1])
        C = C[:-1]
        if opttype == 'P':
            C = np.maximum(C, K-S)
        else:
            C = np.maximum(C, S-K)
    return C[0]
american_option_tree(S0,K,T,r,N,u,d)

np.float64(4.654588754602527)